# Step 3 - Feature Engineering

Lags, rolling aggregates, weather-adjusted pour indicators, inventory turnover. Includes the leakage audit.

In [1]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Add src folder to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project Root:", PROJECT_ROOT)

Project Root: /sessions/admiring-festive-archimedes/mnt/MIG_Cement_Demand_Forecasting


In [2]:
import pandas as pd
import numpy as np

from mig_cement.config import settings
from mig_cement.data import load, validate, preprocess


## Load Clean Data

In [3]:
panel = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")

print(f"Dataset Shape: {panel.shape}")

panel.head()

Dataset Shape: (32880, 22)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,behavior,received_tonnes,rejected_delivery_tonnes,served_tonnes,induced_shortfall,was_constrained,unmet_tonnes,silo_utilisation,cover_days,y
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,aggressive,45.83,0.0,34.54,0,1,8.64,0.142522,1.521714,34.54
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,aggressive,19.97,0.0,45.26,0,0,0.00,0.086071,1.410738,45.26
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,aggressive,47.19,0.0,38.69,0,0,0.00,0.105045,0.996640,38.69
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,aggressive,18.74,0.0,33.16,0,0,0.00,0.072857,1.419180,33.16
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,aggressive,14.40,0.0,47.04,0,1,9.84,0.000000,0.693878,47.04


## Convert the date column and sort the data

In [4]:
# Convert date column to datetime
panel["date"] = pd.to_datetime(panel["date"])

# Sort the data by site and date
panel = panel.sort_values(["site_id", "date"]).reset_index(drop=True)

# Check the first few rows
panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,behavior,received_tonnes,rejected_delivery_tonnes,served_tonnes,induced_shortfall,was_constrained,unmet_tonnes,silo_utilisation,cover_days,y
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,aggressive,45.83,0.0,34.54,0,1,8.64,0.142522,1.521714,34.54
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,aggressive,19.97,0.0,45.26,0,0,0.00,0.086071,1.410738,45.26
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,aggressive,47.19,0.0,38.69,0,0,0.00,0.105045,0.996640,38.69
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,aggressive,18.74,0.0,33.16,0,0,0.00,0.072857,1.419180,33.16
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,aggressive,14.40,0.0,47.04,0,1,9.84,0.000000,0.693878,47.04


## Create Calendar Features

In [5]:
panel["date"] = pd.to_datetime(panel["date"])

panel["year"] = panel["date"].dt.year
panel["month"] = panel["date"].dt.month
panel["week"] = panel["date"].dt.isocalendar().week.astype(int)
panel["day"] = panel["date"].dt.day
panel["day_of_week"] = panel["date"].dt.dayofweek
panel["quarter"] = panel["date"].dt.quarter
panel["is_weekend"] = panel["day_of_week"].isin([5,6]).astype(int)    

panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,silo_utilisation,cover_days,y,year,month,week,day,day_of_week,quarter,is_weekend
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,0.142522,1.521714,34.54,2022,1,52,1,5,1,1
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,0.086071,1.410738,45.26,2022,1,52,2,6,1,1
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,0.105045,0.996640,38.69,2022,1,1,3,0,1,0
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,0.072857,1.419180,33.16,2022,1,1,4,1,1,0
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,0.000000,0.693878,47.04,2022,1,1,5,2,1,0


## Business Feature Engineering

In [6]:
panel["available_cement"] = (
    panel["opening_inventory_tonnes"] +
    panel["deliveries_tonnes"])


panel["storage_utilisation"] = (
    panel["opening_inventory_tonnes"] /
    panel["silo_capacity"])

panel["inventory_buffer"] = (
    panel["available_cement"] -
    panel["planned_pour_tonnes"])


panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,year,month,week,day,day_of_week,quarter,is_weekend,available_cement,storage_utilisation,inventory_buffer
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,2022,1,52,1,5,1,1,98.39,0.117321,55.21
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,2022,1,52,2,6,1,1,83.82,0.142522,38.56
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,2022,1,1,3,0,1,0,85.75,0.086071,47.06
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,2022,1,1,4,1,1,0,65.80,0.105045,32.64
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,2022,1,1,5,2,1,0,47.04,0.072857,-9.84


## Lag Features

In [7]:
# Ensure data is sorted by site and date before creating lag features
panel = panel.sort_values(["site_id", "date"]).reset_index(drop=True)

# Previous day's cement consumption
panel["consumed_lag_1"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .shift(1))

# Cement consumption from one week earlier
panel["consumed_lag_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .shift(7))

panel["consumed_lag_14"] = (
    panel.groupby("site_id")["consumed_tonnes"]
    .shift(14))

panel["consumed_lag_28"] = (
    panel.groupby("site_id")["consumed_tonnes"]
    .shift(28))

# Display the first 10 rows
panel.head(10)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,day_of_week,quarter,is_weekend,available_cement,storage_utilisation,inventory_buffer,consumed_lag_1,consumed_lag_7,consumed_lag_14,consumed_lag_28
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,5,1,1,98.39,0.117321,55.21,NaN,NaN,NaN,NaN
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,6,1,1,83.82,0.142522,38.56,34.54,NaN,NaN,NaN
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,0,1,0,85.75,0.086071,47.06,45.26,NaN,NaN,NaN
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,1,1,0,65.80,0.105045,32.64,38.69,NaN,NaN,NaN
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,2,1,0,47.04,0.072857,-9.84,33.16,NaN,NaN,NaN
5,2022-01-06,SITE_001,CEM_II,30.02,22.19,0.00,22.19,0.00,1.29,14.61,...,3,1,0,22.19,0.000000,-7.83,47.04,NaN,NaN,NaN
6,2022-01-07,SITE_001,CEM_I,33.60,33.60,0.00,47.72,14.12,0.90,4.48,...,4,1,0,47.72,0.000000,14.12,22.19,NaN,NaN,NaN
7,2022-01-08,SITE_001,CEM_I,42.12,34.28,14.12,20.16,0.00,1.95,25.92,...,5,1,1,34.28,0.031518,-7.84,33.60,34.54,NaN,NaN
8,2022-01-09,SITE_001,CEM_I,0.00,0.00,0.00,34.38,34.38,1.42,16.81,...,6,1,1,34.38,0.000000,34.38,34.28,45.26,NaN,NaN
9,2022-01-10,SITE_001,CEM_II,30.99,30.99,34.38,35.37,38.76,3.49,16.57,...,0,1,0,69.75,0.076741,38.76,0.00,38.69,NaN,NaN


### Average Consumption

In [8]:
# 7-day rolling average consumption
panel["rolling_mean_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .transform(lambda x: x.shift(1).rolling(7).mean())
)

# 7-day rolling standard deviation
panel["rolling_std_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .transform(lambda x: x.shift(1).rolling(7).std())
)

panel["rolling_mean_14"] = (
    panel.groupby("site_id")["consumed_tonnes"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
)

panel["rolling_mean_28"] = (
    panel.groupby("site_id")["consumed_tonnes"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

panel.head(10)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,storage_utilisation,inventory_buffer,consumed_lag_1,consumed_lag_7,consumed_lag_14,consumed_lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_mean_28
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,0.117321,55.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,0.142522,38.56,34.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,0.086071,47.06,45.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,0.105045,32.64,38.69,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,0.072857,-9.84,33.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2022-01-06,SITE_001,CEM_II,30.02,22.19,0.00,22.19,0.00,1.29,14.61,...,0.000000,-7.83,47.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2022-01-07,SITE_001,CEM_I,33.60,33.60,0.00,47.72,14.12,0.90,4.48,...,0.000000,14.12,22.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2022-01-08,SITE_001,CEM_I,42.12,34.28,14.12,20.16,0.00,1.95,25.92,...,0.031518,-7.84,33.60,34.54,NaN,NaN,36.354286,8.373171,NaN,NaN
8,2022-01-09,SITE_001,CEM_I,0.00,0.00,0.00,34.38,34.38,1.42,16.81,...,0.000000,34.38,34.28,45.26,NaN,NaN,36.317143,8.383131,NaN,NaN
9,2022-01-10,SITE_001,CEM_II,30.99,30.99,34.38,35.37,38.76,3.49,16.57,...,0.076741,38.76,0.00,38.69,NaN,NaN,29.851429,15.099577,NaN,NaN


In [9]:
# rolling std at 14 and 28 to match the means
for w in [14, 28]:
    panel[f"rolling_std_{w}"] = (
        panel.groupby("site_id")["consumed_tonnes"]
             .transform(lambda x, w=w: x.shift(1).rolling(w).std())
    )

panel[["rolling_mean_7","rolling_std_7","rolling_mean_14","rolling_std_14",
       "rolling_mean_28","rolling_std_28"]].describe().round(2)

,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28
count,32670.00,32670.00,32460.00,32460.00,32040.00,32040.00
mean,23.70,12.99,23.70,13.24,23.69,13.31
std,9.65,7.29,8.86,6.73,8.46,6.48
min,2.34,1.07,6.46,2.05,7.66,3.10
25%,13.19,5.85,12.92,5.68,12.60,5.48
50%,25.69,12.68,26.70,13.76,27.40,14.12
75%,31.22,18.67,30.65,18.54,30.32,18.20
max,57.24,35.70,46.68,32.17,39.88,29.58


## Planned Pour Features

The pour schedule is known ahead, so forward-looking sums are legal here in a way
they would not be for any target-derived column.

In [10]:
def forward_sum(s, window):
    return s.iloc[::-1].rolling(window, min_periods=1).sum().iloc[::-1]

for w in [7, 14, 28, 56]:
    panel[f"planned_pour_next_{w}"] = (
        panel.groupby("site_id")["planned_pour_tonnes"]
             .transform(lambda s, w=w: forward_sum(s, w))
    )

panel["planned_pour_lag_1"] = panel.groupby("site_id")["planned_pour_tonnes"].shift(1)
panel["planned_vs_rolling_mean"] = panel["planned_pour_tonnes"] / panel["rolling_mean_7"]

panel[[c for c in panel.columns if c.startswith("planned_pour_next")]].head()

,planned_pour_next_7,planned_pour_next_14,planned_pour_next_28,planned_pour_next_56
0,280.79,521.48,1202.89,2432.32
1,279.73,528.57,1216.03,2441.25
2,234.47,521.05,1209.58,2448.84
3,226.77,539.28,1219.73,2466.91
4,230.87,557.13,1243.27,2474.87


In [11]:
# days since the last scheduled pour (schedule is known ahead)
pour_day = panel["date"].where(panel["planned_pour_tonnes"] > 0)
last_pour = pour_day.groupby(panel["site_id"]).ffill()
panel["days_since_planned_pour"] = (panel["date"] - last_pour).dt.days

# days since last actual consumption - target-derived, so shifted
consumed_day = panel["date"].where(panel["consumed_tonnes"] > 0)
last_consumed = consumed_day.groupby(panel["site_id"]).shift(1).groupby(panel["site_id"]).ffill()
panel["days_since_consumption"] = (panel["date"] - last_consumed).dt.days

panel[["days_since_planned_pour","days_since_consumption"]].describe().round(2)

,days_since_planned_pour,days_since_consumption
count,32879.00,32848.00
mean,0.11,1.15
std,0.37,0.44
min,0.00,1.00
25%,0.00,1.00
50%,0.00,1.00
75%,0.00,1.00
max,5.00,6.00


## Weather Features

In [12]:
# Thresholds are empirical, not guesses - see the band tables below.
# Rain: a hard step at 15 mm. Temperature: a step at 0 C.
panel["rain_mm_capped"] = panel["rain_mm"].clip(upper=30)
panel["pour_blocked_rain"] = (panel["rain_mm"] > 15).astype(int)
panel["frost"] = (panel["avg_temp_c"] < 0).astype(int)

panel["temp_bin"] = pd.cut(panel["avg_temp_c"], bins=[-np.inf, 0, 10, 20, np.inf],
                           labels=["frost", "cold", "mild", "warm"])

rain_bands = pd.cut(panel["rain_mm"], [-.01, 5, 10, 14, 15, 16, 20, 50])
panel.groupby(rain_bands, observed=True).agg(
    n=("consumed_tonnes", "size"),
    pct_zero=("consumed_tonnes", lambda s: round(100 * (s == 0).mean(), 1)),
    mean_consumed=("consumed_tonnes", "mean"),
    mean_planned=("planned_pour_tonnes", "mean"),
).round(2)

,n,pct_zero,mean_consumed,mean_planned
rain_mm,,,,
"(-0.01, 5.0]",20786,9.4,24.70,30.62
"(5.0, 10.0]",7665,9.1,24.80,30.83
"(10.0, 14.0]",2443,9.3,25.07,30.91
"(14.0, 15.0]",359,8.6,24.56,30.90
"(15.0, 16.0]",314,68.2,3.95,31.49
"(16.0, 20.0]",694,68.0,3.89,30.14
"(20.0, 50.0]",619,67.5,4.00,30.79


In [13]:
temp_bands = pd.cut(panel["avg_temp_c"], [-np.inf, -1, 0, 1, 5, 15, np.inf])
panel.groupby(temp_bands, observed=True).agg(
    n=("consumed_tonnes", "size"),
    pct_zero=("consumed_tonnes", lambda s: round(100 * (s == 0).mean(), 1)),
    mean_consumed=("consumed_tonnes", "mean"),
).round(2)

,n,pct_zero,mean_consumed
avg_temp_c,,,
"(-inf, -1.0]",3643,11.7,19.86
"(-1.0, 0.0]",927,12.1,19.96
"(0.0, 1.0]",1032,13.4,24.64
"(1.0, 5.0]",4663,11.5,25.28
"(5.0, 15.0]",12319,12.2,24.29
"(15.0, inf]",10296,12.5,23.94


Two different mechanisms, so two different features.

Above 15 mm rain the pour is abandoned: zero-consumption jumps from 9% to 68% and
mean volume falls from 25.7 t to 3.7 t. Below 15 mm rain has no effect at all — the
0–15 mm range is flat. The break is a step, not a gradient, so `pour_blocked_rain`
is a flag rather than a scaled value.

Below 0 C the pour still happens but is smaller: mean volume drops from 25.4 t to
20.7 t while the zero rate stays at ~12%. Frost reduces throughput, it does not
cancel work.

Rain correlates -0.18 with consumption but 0.001 with `planned_pour_tonnes`, so
weather never changes the schedule — it disrupts execution. That means
`was_constrained` mixes two causes, stockouts and weather cancellations, which
matters when the censoring treatment is decided in Step 4.

## Site Attributes and Inventory Position

In [14]:
panel["region"] = panel["region"].astype("category")
panel["behavior"] = panel["behavior"].astype("category")
panel["cement_type"] = panel["cement_type"].astype("category")

# cover days from opening stock against recent demand
panel["cover_days_7"] = panel["opening_inventory_tonnes"] / panel["rolling_mean_7"]
panel["cover_days_7"] = panel["cover_days_7"].replace([np.inf, -np.inf], np.nan)

panel["inventory_vs_capacity"] = panel["opening_inventory_tonnes"] / panel["silo_capacity"]
panel["headroom_tonnes"] = panel["silo_capacity"] - panel["opening_inventory_tonnes"]

panel[["cover_days_7","inventory_vs_capacity","headroom_tonnes"]].describe().round(2)

,cover_days_7,inventory_vs_capacity,headroom_tonnes
count,32670.00,32880.00,32880.00
mean,9.67,0.39,193.92
std,13.98,0.42,156.06
min,0.00,0.00,0.00
25%,0.01,0.00,17.08
50%,1.63,0.15,180.00
75%,15.39,0.93,314.00
max,160.48,1.00,487.00


## Leakage Audit

Two things are checked. First, that no feature at time *t* moves when a future value
changes. Second, when each feature is actually available, since an 8-week forecast
cannot use values that only exist after the fact.

In [15]:
TARGET = "y"
ID_COLS = ["date", "site_id"]
EXCLUDE = [TARGET, "consumed_tonnes", "served_tonnes", "closing_inventory_tonnes",
           "silo_utilisation", "cover_days", "was_constrained", "unmet_tonnes",
           "induced_shortfall", "rejected_delivery_tonnes", "received_tonnes",
           "deliveries_tonnes", "available_cement", "inventory_buffer"]

feature_cols = [c for c in panel.columns if c not in EXCLUDE + ID_COLS]
print(f"{len(feature_cols)} features")
print(sorted(feature_cols))

41 features
['avg_temp_c', 'behavior', 'cement_type', 'consumed_lag_1', 'consumed_lag_14', 'consumed_lag_28', 'consumed_lag_7', 'cover_days_7', 'day', 'day_of_week', 'days_since_consumption', 'days_since_planned_pour', 'frost', 'headroom_tonnes', 'inventory_vs_capacity', 'is_weekend', 'month', 'opening_inventory_tonnes', 'planned_pour_lag_1', 'planned_pour_next_14', 'planned_pour_next_28', 'planned_pour_next_56', 'planned_pour_next_7', 'planned_pour_tonnes', 'planned_vs_rolling_mean', 'pour_blocked_rain', 'quarter', 'rain_mm', 'rain_mm_capped', 'region', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_7', 'rolling_std_14', 'rolling_std_28', 'rolling_std_7', 'silo_capacity', 'storage_utilisation', 'temp_bin', 'week', 'year']


`deliveries_tonnes` is dropped, and with it `available_cement` and
`inventory_buffer`, which were both built from it. Actual deliveries on day *t* are
not known when forecasting day *t* eight weeks out — only the opening position and
the pour schedule are. `planned_pour_tonnes` stays because it is scheduled.

In [16]:
def rebuild(df):
    """Recompute the target-derived features on a copy."""
    d = df.sort_values(["site_id", "date"]).copy()
    g = d.groupby("site_id")["consumed_tonnes"]
    for lag in [1, 7, 14, 28]:
        d[f"consumed_lag_{lag}"] = g.shift(lag)
    for w in [7, 14, 28]:
        d[f"rolling_mean_{w}"] = g.transform(lambda x, w=w: x.shift(1).rolling(w).mean())
        d[f"rolling_std_{w}"] = g.transform(lambda x, w=w: x.shift(1).rolling(w).std())
    return d


derived = [c for c in feature_cols if c.startswith(("consumed_lag", "rolling_"))]

base = rebuild(panel)
mutated = panel.copy()
last = mutated.index[-1]
mutated.loc[last, "consumed_tonnes"] = 99999.0
after = rebuild(mutated)

changed = [c for c in derived
           if not base[c].iloc[:-1].equals(after[c].iloc[:-1])]
print("features altered by a future target value:", changed or "none")
assert not changed, "leakage detected"


features altered by a future target value: none


In [17]:
availability = pd.DataFrame([
    ("calendar (year, month, week, day_of_week, quarter, is_weekend)", "always", "deterministic"),
    ("site attributes (region, behavior, silo_capacity, cement_type)", "always", "static"),
    ("planned_pour_tonnes, planned_pour_next_*, days_since_planned_pour", "always", "schedule known ahead"),
    ("consumed_lag_*, rolling_*, days_since_consumption", "h=1 only", "recursive beyond one step"),
    ("rain_*, temp_*", "~14 days", "needs a weather forecast"),
    ("opening_inventory, cover_days_7, inventory_vs_capacity, headroom", "h=1 only", "from the Phase 4 simulation"),
], columns=["feature group", "available at", "note"])
availability

,feature group,available at,note
0,"calendar (year, month, week, day_of_week, quar...",always,deterministic
1,"site attributes (region, behavior, silo_capaci...",always,static
2,"planned_pour_tonnes, planned_pour_next_*, days...",always,schedule known ahead
3,"consumed_lag_*, rolling_*, days_since_consumption",h=1 only,recursive beyond one step
4,"rain_*, temp_*",~14 days,needs a weather forecast
5,"opening_inventory, cover_days_7, inventory_vs_...",h=1 only,from the Phase 4 simulation


Only the first three groups are available across the full 8-week horizon. Lags,
rolling statistics and inventory position have to be produced recursively at
inference, and weather beyond roughly two weeks is not obtainable — which is why
`build_features()` in `src/` takes a `for_inference` flag.

## Save the Feature Matrix

In [18]:
matrix = panel[ID_COLS + feature_cols + [TARGET]].copy()

print(f"shape: {matrix.shape}")
print(f"rows with any NaN: {matrix.isna().any(axis=1).sum():,} "
      f"({matrix.isna().any(axis=1).mean():.1%}) - warm-up period for the 28-day windows")

dest = settings.processed_dir / "operations_feature_engineered.parquet"
dest.parent.mkdir(parents=True, exist_ok=True)
matrix.to_parquet(dest, index=False)
print(f"\nwrote {len(matrix):,} rows -> {dest.name}")

shape: (32880, 44)
rows with any NaN: 840 (2.6%) - warm-up period for the 28-day windows



wrote 32,880 rows -> operations_feature_engineered.parquet


In [19]:
matrix.dtypes.to_frame("dtype")

,dtype
date,datetime64[ns]
site_id,object
cement_type,category
planned_pour_tonnes,float64
opening_inventory_tonnes,float64
rain_mm,float64
avg_temp_c,float64
silo_capacity,int64
region,category
behavior,category
